# Initial EDA: United States Top 50 Playlist

This notebook profiles the supplied dataset. It does not modify the raw CSV. Update `DATA_PATH` if your file is stored elsewhere. Review the quality report before interpreting unusual dates.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'Atlantic_United_States.csv'
raw = pd.read_csv(DATA_PATH)
print(f'Loaded {len(raw):,} rows from {DATA_PATH}')

## Structure and quality

In [ ]:
display(raw.head())
display(raw.dtypes.rename('dtype').to_frame())
display(raw.isna().sum().rename('missing').to_frame())
print('Exact duplicate rows:', raw.duplicated().sum())
print('Columns:', raw.columns.tolist())

In [ ]:
if 'date' in raw.columns:
    dates = pd.to_datetime(raw['date'], format='%d-%m-%Y', errors='coerce')
    print('Date range:', dates.min(), 'to', dates.max())
    print('Unparseable dates:', dates.isna().sum())
    print('Rows per date:')
    display(dates.value_counts().sort_index().describe().to_frame('rows_per_date'))
if {'date', 'position'}.issubset(raw.columns):
    keys = pd.DataFrame({'date': dates, 'position': raw['position']})
    print('Duplicate date-position keys:', keys.duplicated().sum())

## Chart distribution and rank profile

In [ ]:
if 'position' in raw.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.histplot(raw['position'].dropna(), bins=range(1, 52), discrete=True, ax=ax)
    ax.set(title='Observed chart positions', xlabel='Position (1 = highest)', ylabel='Observations')
    plt.show()
if 'artist' in raw.columns:
    display(raw['artist'].value_counts().head(15).rename_axis('artist').to_frame('chart_appearances'))
if 'album_type' in raw.columns:
    display(raw['album_type'].value_counts(dropna=False).rename_axis('album_type').to_frame('rows'))

## Notes and next steps

Record the source, collection method, unusual keys, and interpretation limits in `reports/data_quality_report.md`. Do not resolve conflicting chart ranks by guessing. Continue with song longevity and ranking timelines after validating the date-position keys.